<a href="https://colab.research.google.com/github/9terry-student/Transformer-based_LLM_from_scratch/blob/main/numpy_transformer_no_hints.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import numpy as np

def softmax(x):
  output=np.exp(x)/np.sum(np.exp(x),axis=-1,keepdims=True)
  return output

def attention(q,k,v,mask=None):
  scores=q@k.T
  if mask!=None:
    m=np.full((seq_len,seq_len),-np.inf)
    m=np.triu(m,k=1)
    scores+=m
  weights=softmax(scores/np.sqrt(d_k))
  output=weights@v
  return output,weights

def multihead(q,k,v,mask=None):
  w_q=[np.random.randn(d_model,d_k) for i in range(num_heads)]
  w_k=[np.random.randn(d_model,d_k) for i in range(num_heads)]
  w_v=[np.random.randn(d_model,d_k) for i in range(num_heads)]
  w_o=np.random.randn(d_model,d_model)
  heads=[]
  for i in range(num_heads):
    head,_=attention(q@w_q[i],k@w_k[i],v@w_v[i])
    heads.append(head)
  output=np.concatenate(heads,axis=-1)@w_o
  return output

def ffn(x):
  w1=np.random.randn(d_model,d_ff)
  w2=np.random.randn(d_ff,d_model)
  b1=np.zeros(d_ff)
  b2=np.zeros(d_model)
  output=np.maximum(0,x@w1+b1)@w2+b2
  return output

def layernorm(x):
  r=np.ones(d_model)
  b=np.zeros(d_model)
  output=((x-np.mean(x,axis=-1,keepdims=True))*r/(np.std(x,axis=-1,keepdims=True)+10**(-9))+b)
  return output

def pe(pos,i):
  if i%2==0:
    output=np.sin(pos/10**(4*i/d_model))
    return output
  output=np.cos(pos/10**(4*(i-1)/d_model))
  return output

In [21]:
np.random.seed(42)
seq_len = 4
d_model = 16
d_k = 8
d_ff = 64
num_heads = 2

x = np.random.randn(seq_len, d_model)
out = multihead(x, x, x)
print("multihead shape:", out.shape)  # (4, 16)

out = ffn(x)
print("ffn shape:", out.shape)  # (4, 16)

out = layernorm(x)
print("layernorm shape:", out.shape)  # (4, 16)

Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

_, weights = attention(Q, K, V)
print("normal weights:\n", weights.round(3))

_, weights_masked = attention(Q, K, V, mask=True)
print("masked weights:\n", weights_masked.round(3))

multihead shape: (4, 16)
ffn shape: (4, 16)
layernorm shape: (4, 16)
normal weights:
 [[0.032 0.063 0.008 0.897]
 [0.112 0.052 0.087 0.75 ]
 [0.338 0.264 0.309 0.089]
 [0.227 0.107 0.071 0.594]]
masked weights:
 [[1.    0.    0.    0.   ]
 [0.685 0.315 0.    0.   ]
 [0.371 0.289 0.34  0.   ]
 [0.227 0.107 0.071 0.594]]


In [22]:
def encoder(x):
  output=layernorm(x+multihead(x,x,x))
  output=layernorm(output+ffn(output))
  return output

def decoder(x,enc_output):
  output=layernorm(x+multihead(x,x,x,mask=True))
  output=layernorm(output+multihead(output,enc_output,enc_output))
  output=layernorm(output+ffn(output))
  return output

In [23]:
x = np.random.randn(seq_len, d_model)
pe_matrix = np.array([[pe(pos,i) for i in range(d_model)] for pos in range(seq_len)])

enc_output = encoder(x + pe_matrix)
dec_output = decoder(x + pe_matrix, enc_output)

print("encoder output shape:", enc_output.shape)  # (4, 16)
print("decoder output shape:", dec_output.shape)  # (4, 16)

encoder output shape: (4, 16)
decoder output shape: (4, 16)
